# Reproduce best Testing results from an embedded model list

All code below is in English and runs inside the notebook.
No external `all_model_results_detailed.csv` input is required; the Testing model list
is embedded in this notebook.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any, Dict, Iterable

import numpy as np
import pandas as pd

from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.utils.class_weight import compute_sample_weight

from src.config import CATEGORICAL_COLUMNS, LABELS, METRICS_DIR, RANDOM_STATE, TARGET_COL
from src.evaluation import classification_metrics
from src.preprocessing import split_feature_columns


In [ ]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = Path('/home/husrcf/Code/ProtBind/ML')

TRAIN_PATH = PROJECT_ROOT / 'data' / 'processed' / 'train.csv'
TEST_PATH = PROJECT_ROOT / 'data' / 'processed' / 'test.csv'
OUT_DIR = PROJECT_ROOT / 'models' / 'metrics' / 'reproduction_from_notebook'
OUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_SET = 'Testing'
NOT_DROPOUT = 'NotDropout'
WEIGHT_STEP = 0.05

EMBEDDED_TESTING_MODELS = [
    {'Set': 'Testing', 'Model': 'catboost_cumulative_test_stage_a_dropout_vs_not', 'F1 (macro)': 0.8512044817927171, 'AUC': np.nan},
    {'Set': 'Testing', 'Model': 'lightgbm_hist_cumulative_test_stage_a_dropout_vs_not', 'F1 (macro)': 0.8473345384200679, 'AUC': np.nan},
    {'Set': 'Testing', 'Model': 'gbdt_cumulative_test_stage_a_dropout_vs_not', 'F1 (macro)': 0.8442837600156341, 'AUC': np.nan},
    {'Set': 'Testing', 'Model': 'catboost_cumulative_test_stage_b_enrolled_vs_graduate', 'F1 (macro)': 0.8017637124779982, 'AUC': np.nan},
    {'Set': 'Testing', 'Model': 'lightgbm_hist_cumulative_test_stage_b_enrolled_vs_graduate', 'F1 (macro)': 0.794536858908137, 'AUC': np.nan},
    {'Set': 'Testing', 'Model': 'gbdt_cumulative_test_stage_b_enrolled_vs_graduate', 'F1 (macro)': 0.7930733544623582, 'AUC': np.nan},
    {'Set': 'Testing', 'Model': 'gbdt_log_loss', 'F1 (macro)': 0.7501419915234154, 'AUC': 0.9057734233652438},
    {'Set': 'Testing', 'Model': 'catboost', 'F1 (macro)': 0.7272921202793698, 'AUC': np.nan},
    {'Set': 'Testing', 'Model': 'catboost_tuned', 'F1 (macro)': 0.7222170816610584, 'AUC': 0.8959441300222126},
    {'Set': 'Testing', 'Model': 'catboost_tuned_raw_test', 'F1 (macro)': 0.7222170816610584, 'AUC': 0.8959441300222126},
    {'Set': 'Testing', 'Model': 'catboost_tuned_test', 'F1 (macro)': 0.7222170816610584, 'AUC': 0.8959441300222126},
    {'Set': 'Testing', 'Model': 'catboost_tuned_biased_test', 'F1 (macro)': 0.7216502445804736, 'AUC': 0.897300221017038},
    {'Set': 'Testing', 'Model': 'lightgbm_tuned_semester_test', 'F1 (macro)': 0.7214186399470116, 'AUC': 0.8925942777011562},
    {'Set': 'Testing', 'Model': 'lightgbm_hist_column_best_test', 'F1 (macro)': 0.7196343259456159, 'AUC': 0.8921260456696501},
    {'Set': 'Testing', 'Model': 'lightgbm_hist_column_best', 'F1 (macro)': 0.7196343259456159, 'AUC': np.nan},
    {'Set': 'Testing', 'Model': 'lightgbm', 'F1 (macro)': 0.7196343259456159, 'AUC': 0.8921260456696501},
    {'Set': 'Testing', 'Model': 'lightgbm_hist_column_best_seed42_test', 'F1 (macro)': 0.7196343259456159, 'AUC': 0.8921260456696501},
    {'Set': 'Testing', 'Model': 'catboost_tuned_semester_test', 'F1 (macro)': 0.7170676031016795, 'AUC': 0.8963778513814473},
    {'Set': 'Testing', 'Model': 'catboost_tuned_seed42_test', 'F1 (macro)': 0.7128226177576588, 'AUC': 0.8935675868359864},
    {'Set': 'Testing', 'Model': 'gbdt_log_loss_seed82_test', 'F1 (macro)': 0.7114207642329795, 'AUC': 0.8878150465525935},
    {'Set': 'Testing', 'Model': 'gbdt_log_loss_seed42_test', 'F1 (macro)': 0.7111756258599903, 'AUC': 0.8883128004959436},
    {'Set': 'Testing', 'Model': 'gbdt_log_loss_test', 'F1 (macro)': 0.7111756258599903, 'AUC': 0.8883128004959436},
    {'Set': 'Testing', 'Model': 'catboost_cumulative_test', 'F1 (macro)': 0.7110834705072161, 'AUC': 0.8896189758083727},
    {'Set': 'Testing', 'Model': 'lightgbm_hist_column_best_seed62_test', 'F1 (macro)': 0.7106101757920235, 'AUC': 0.8912203259862982},
    {'Set': 'Testing', 'Model': 'gbdt_log_loss_seed62_test', 'F1 (macro)': 0.710590086019346, 'AUC': 0.8879798406146509},
    {'Set': 'Testing', 'Model': 'gbdt_cumulative_test', 'F1 (macro)': 0.7102694589968809, 'AUC': 0.8790695586967209},
    {'Set': 'Testing', 'Model': 'catboost_tuned_seed62_test', 'F1 (macro)': 0.7095073231495185, 'AUC': 0.8928003103757352},
    {'Set': 'Testing', 'Model': 'catboost_tuned_seed82_test', 'F1 (macro)': 0.7087551749650897, 'AUC': 0.8923366760683953},
    {'Set': 'Testing', 'Model': 'cumulative_link_ensemble_test', 'F1 (macro)': 0.7083193880713963, 'AUC': 0.8925199720169594},
    {'Set': 'Testing', 'Model': 'lightgbm_hist_cumulative_test', 'F1 (macro)': 0.7078484886346895, 'AUC': 0.8900586871558},
    {'Set': 'Testing', 'Model': 'lightgbm_hist_column_best_seed82_test', 'F1 (macro)': 0.7069540179235135, 'AUC': 0.8914655534767877},
    {'Set': 'Testing', 'Model': 'lightgbm_tuned_test', 'F1 (macro)': 0.7048030677935238, 'AUC': 0.8883239131941032},
    {'Set': 'Testing', 'Model': 'lightgbm_tuned', 'F1 (macro)': 0.7048030677935238, 'AUC': np.nan},
]

EMBEDDED_TESTING_MODELS_DF = pd.DataFrame(EMBEDDED_TESTING_MODELS)

FALLBACK_BEST_PARAMS: Dict[str, Dict[str, Any]] = {
    'catboost_tuned': {
        'depth': 6,
        'iterations': 500,
        'l2_leaf_reg': 9,
        'learning_rate': 0.05,
        'random_strength': 1.0,
    },
    'lightgbm_tuned': {
        'colsample_bytree': 1.0,
        'learning_rate': 0.03,
        'max_depth': -1,
        'min_child_samples': 40,
        'n_estimators': 300,
        'num_leaves': 63,
        'subsample': 0.8,
    },
    'hist_column_sampling': {
        'max_bin': 16,
        'column_fraction': 0.6,
    },
    'gbdt_log_loss': {
        'learning_rate': 0.1,
        'max_depth': 3,
        'n_estimators': 400,
        'subsample': 1.0,
    },
}

BEST_PARAM_FILES = {
    'catboost_tuned': PROJECT_ROOT / 'models' / 'metrics' / 'catboost_tuned_best_params.json',
    'lightgbm_tuned': PROJECT_ROOT / 'models' / 'metrics' / 'lightgbm_tuned_best_params.json',
    'hist_column_sampling': PROJECT_ROOT / 'models' / 'metrics' / 'hist_column_sampling_best_params.json',
    'gbdt_log_loss': PROJECT_ROOT / 'models' / 'metrics' / 'classic_boosting_best_params.json',
}


def load_processed_data():
    train_df = pd.read_csv(TRAIN_PATH)
    test_df = pd.read_csv(TEST_PATH)
    if TARGET_COL not in train_df.columns or TARGET_COL not in test_df.columns:
        raise RuntimeError(f"Missing target column '{TARGET_COL}' in processed datasets.")
    return train_df, test_df


def _dense_preprocessor(num_features, cat_features):
    num_pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )
    cat_pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]
    )
    return ColumnTransformer(
        transformers=[
            ("num", num_pipe, num_features),
            ("cat", cat_pipe, cat_features),
        ],
        sparse_threshold=0.0,
    )


def prepare_catboost_xy(df: pd.DataFrame):
    X = df.drop(columns=[TARGET_COL]).copy()
    y = df[TARGET_COL].astype(str).copy()
    cat_cols = [col for col in X.columns if col in CATEGORICAL_COLUMNS]
    cat_idx = [X.columns.get_loc(col) for col in cat_cols]
    for c in cat_cols:
        X[c] = X[c].astype(str)
    return X, y, cat_idx


def prepare_lightgbm_xy(df: pd.DataFrame):
    X = df.drop(columns=[TARGET_COL]).copy()
    y = df[TARGET_COL].astype(str).copy()
    cat_cols = [col for col in X.columns if col in CATEGORICAL_COLUMNS]
    for c in cat_cols:
        X[c] = X[c].astype("category")
    return X, y, cat_cols


def _binary_proba(model, X, positive_label: str):
    probs = model.predict_proba(X)
    idx_map = {str(v): i for i, v in enumerate(model.classes_)}
    return np.asarray(probs[:, idx_map[str(positive_label)]], dtype=float)


def _combine_binary(prob_not_dropout, prob_grad):
    p_not_dropout = np.clip(np.asarray(prob_not_dropout, dtype=float), 0.0, 1.0)
    p_grad = np.clip(np.asarray(prob_grad, dtype=float), 0.0, 1.0)
    proba = np.column_stack([
        1.0 - p_not_dropout,
        p_not_dropout * (1.0 - p_grad),
        p_not_dropout * p_grad,
    ])
    row_sum = proba.sum(axis=1, keepdims=True)
    return proba / np.where(row_sum == 0.0, 1.0, row_sum)


def _eval_prob(name: str, y_true, proba: np.ndarray):
    y_true = y_true.astype(str)
    labels = list(LABELS)
    y_pred = np.array([labels[i] for i in np.argmax(proba, axis=1)])
    m = classification_metrics(y_true, y_pred, y_prob=proba, labels=labels)
    m2: Dict[str, float] = {
        'model': name,
        'accuracy': float(m.get('accuracy', np.nan)),
        'f1_macro': float(m.get('f1_macro', np.nan)),
        'f1_weighted': float(m.get('f1_weighted', np.nan)),
        'auc': float(m.get('roc_auc_ovr_macro', np.nan)),
        'dropout_f1': float(m.get('dropout_f1', np.nan)),
        'enrolled_f1': float(m.get('enrolled_f1', np.nan)),
        'graduate_f1': float(m.get('graduate_f1', np.nan)),
    }
    return m2


def _eval_stage_a(name: str, y_true, proba: np.ndarray):
    y_true = y_true.astype(str).to_numpy()
    p_not_dropout = proba[:, 1] + proba[:, 2]
    y_stage = np.where(y_true == 'Dropout', 'Dropout', NOT_DROPOUT)
    y_pred = np.where(p_not_dropout >= 0.5, NOT_DROPOUT, 'Dropout')
    f1 = f1_score(y_stage, y_pred, average='macro', zero_division=0)
    try:
        auc = roc_auc_score((y_stage == NOT_DROPOUT).astype(int), p_not_dropout)
    except ValueError:
        auc = float('nan')
    return {'model': name, 'positive_label': NOT_DROPOUT, 'f1_macro': float(f1), 'auc': float(auc)}


def _eval_stage_b(name: str, y_true, proba: np.ndarray):
    y_true = y_true.astype(str).to_numpy()
    mask = y_true != 'Dropout'
    if not mask.any():
        return {'model': name, 'positive_label': 'Graduate', 'f1_macro': 0.0, 'auc': float('nan')}

    p_not_dropout = proba[:, 1] + proba[:, 2]
    p_grad_given_not = np.zeros_like(p_not_dropout, dtype=float)
    denom = np.clip(p_not_dropout[mask], 1e-12, 1.0)
    p_grad_given_not[mask] = proba[mask, 2] / denom
    y_sub = y_true[mask]
    y_pred = np.where(p_grad_given_not[mask] >= 0.5, 'Graduate', 'Enrolled')
    f1 = f1_score(y_sub, y_pred, average='macro', zero_division=0)
    try:
        auc = roc_auc_score((y_sub == 'Graduate').astype(int), p_grad_given_not[mask])
    except ValueError:
        auc = float('nan')
    return {'model': name, 'positive_label': 'Graduate', 'f1_macro': float(f1), 'auc': float(auc)}


def choose_target_row(model_name: str | None, target_set: str = TARGET_SET, rank_metric: str = 'F1 (macro)'):
    df = EMBEDDED_TESTING_MODELS_DF.copy()
    subset = df[df['Set'] == target_set].copy()
    if subset.empty:
        raise RuntimeError(f'No rows with Set={target_set}')
    if model_name:
        subset = subset[subset['Model'] == model_name].copy()
        if subset.empty:
            raise ValueError(f'Model {model_name} is not available in the embedded testing model list.')
    subset[rank_metric] = pd.to_numeric(subset[rank_metric], errors='coerce')
    return subset.sort_values(rank_metric, ascending=False).iloc[0]


def load_best_params(name: str) -> Dict[str, Any]:
    key = name
    if name.startswith('catboost'):
        key = 'catboost_tuned'
    elif name.startswith('lightgbm_tuned'):
        key = 'lightgbm_tuned'
    elif name.startswith('lightgbm_hist') or name == 'lightgbm':
        key = 'hist_column_sampling'
    elif name.startswith('gbdt_log_loss'):
        key = 'gbdt_log_loss'

    fallback = FALLBACK_BEST_PARAMS.get(key)
    path = BEST_PARAM_FILES.get(key)
    if path is not None and path.exists():
        payload = json.loads(path.read_text(encoding='utf-8'))
        if key == 'hist_column_sampling':
            return payload['models']['lightgbm']['params']
        if key == 'gbdt_log_loss':
            raw = payload['models']['gbdt_log_loss']['best_params']
            return {k.removeprefix('model__'): v for k, v in raw.items()}
        return payload
    if fallback is not None:
        return dict(fallback)
    raise KeyError(f'Unsupported model family for params: {name}')


In [ ]:
def fit_predict_cum_catboost(train_df: pd.DataFrame, test_df: pd.DataFrame, params: Dict[str, Any], cv: int):
    X, y, cat_idx = prepare_catboost_xy(train_df)
    X_test, y_test, _ = prepare_catboost_xy(test_df)

    skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=RANDOM_STATE)
    p_not_dropout_oof = np.zeros(len(train_df), dtype=float)
    p_grad_oof = np.zeros(len(train_df), dtype=float)

    for train_idx, valid_idx in skf.split(X, y):
        y_a = np.where(y.iloc[train_idx].to_numpy() == 'Dropout', 'Dropout', NOT_DROPOUT)
        model_a = CatBoostClassifier(
            loss_function='Logloss',
            eval_metric='F1',
            auto_class_weights='Balanced',
            verbose=0,
            thread_count=-1,
            random_seed=RANDOM_STATE,
            **params,
        )
        model_a.fit(X.iloc[train_idx], y_a, cat_features=cat_idx)
        p_not_dropout_oof[valid_idx] = _binary_proba(model_a, X.iloc[valid_idx], NOT_DROPOUT)

        stage_train_idx = [i for i in train_idx if y.iloc[i] != 'Dropout']
        model_b = CatBoostClassifier(
            loss_function='Logloss',
            eval_metric='F1',
            auto_class_weights='Balanced',
            verbose=0,
            thread_count=-1,
            random_seed=RANDOM_STATE + 1000,
            **params,
        )
        model_b.fit(X.iloc[stage_train_idx], y.iloc[stage_train_idx], cat_features=cat_idx)
        p_grad_oof[valid_idx] = _binary_proba(model_b, X.iloc[valid_idx], 'Graduate')

    full_a = CatBoostClassifier(
        loss_function='Logloss',
        eval_metric='F1',
        auto_class_weights='Balanced',
        verbose=0,
        thread_count=-1,
        random_seed=RANDOM_STATE,
        **params,
    )
    full_a.fit(X, np.where(y.to_numpy() == 'Dropout', 'Dropout', NOT_DROPOUT), cat_features=cat_idx)
    p_not_dropout_test = _binary_proba(full_a, X_test, NOT_DROPOUT)

    full_b = CatBoostClassifier(
        loss_function='Logloss',
        eval_metric='F1',
        auto_class_weights='Balanced',
        verbose=0,
        thread_count=-1,
        random_seed=RANDOM_STATE + 1000,
        **params,
    )
    mask = y.to_numpy() != 'Dropout'
    full_b.fit(X.loc[mask], y.loc[mask], cat_features=cat_idx)
    p_grad_test = _binary_proba(full_b, X_test, 'Graduate')

    return _combine_binary(p_not_dropout_oof, p_grad_oof), _combine_binary(p_not_dropout_test, p_grad_test), y_test


def fit_predict_cum_lightgbm(train_df: pd.DataFrame, test_df: pd.DataFrame, params: Dict[str, Any], cv: int, n_jobs: int):
    X, y, cat_cols = prepare_lightgbm_xy(train_df)
    X_test, y_test, _ = prepare_lightgbm_xy(test_df)

    skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=RANDOM_STATE)
    p_not_dropout_oof = np.zeros(len(train_df), dtype=float)
    p_grad_oof = np.zeros(len(train_df), dtype=float)

    for train_idx, valid_idx in skf.split(X, y):
        y_a = pd.Series(np.where(y.iloc[train_idx].to_numpy() == 'Dropout', 'Dropout', NOT_DROPOUT), index=y.index[train_idx])
        model_a = LGBMClassifier(
            objective='binary',
            class_weight='balanced',
            random_state=RANDOM_STATE,
            verbosity=-1,
            n_jobs=n_jobs,
            **params,
        )
        model_a.fit(X.iloc[train_idx], y_a, categorical_feature=cat_cols)
        p_not_dropout_oof[valid_idx] = _binary_proba(model_a, X.iloc[valid_idx], NOT_DROPOUT)

        stage_train_idx = [i for i in train_idx if y.iloc[i] != 'Dropout']
        model_b = LGBMClassifier(
            objective='binary',
            class_weight='balanced',
            random_state=RANDOM_STATE + 1000,
            verbosity=-1,
            n_jobs=n_jobs,
            **params,
        )
        model_b.fit(X.iloc[stage_train_idx], y.iloc[stage_train_idx], categorical_feature=cat_cols)
        p_grad_oof[valid_idx] = _binary_proba(model_b, X.iloc[valid_idx], 'Graduate')

    full_a = LGBMClassifier(
        objective='binary',
        class_weight='balanced',
        random_state=RANDOM_STATE,
        verbosity=-1,
        n_jobs=n_jobs,
        **params,
    )
    full_a.fit(X, np.where(y.to_numpy() == 'Dropout', 'Dropout', NOT_DROPOUT), categorical_feature=cat_cols)
    p_not_dropout_test = _binary_proba(full_a, X_test, NOT_DROPOUT)

    full_b = LGBMClassifier(
        objective='binary',
        class_weight='balanced',
        random_state=RANDOM_STATE + 1000,
        verbosity=-1,
        n_jobs=n_jobs,
        **params,
    )
    mask = y.to_numpy() != 'Dropout'
    full_b.fit(X.loc[mask], y.loc[mask], categorical_feature=cat_cols)
    p_grad_test = _binary_proba(full_b, X_test, 'Graduate')

    return _combine_binary(p_not_dropout_oof, p_grad_oof), _combine_binary(p_not_dropout_test, p_grad_test), y_test


def fit_predict_cum_gbdt(train_df: pd.DataFrame, test_df: pd.DataFrame, params: Dict[str, Any], cv: int):
    X_train = train_df.drop(columns=[TARGET_COL])
    y_train = train_df[TARGET_COL].astype(str)
    X_test = test_df.drop(columns=[TARGET_COL])
    y_test = test_df[TARGET_COL].astype(str)

    num_feats, cat_feats = split_feature_columns(X_train.columns)
    skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=RANDOM_STATE)
    p_not_dropout_oof = np.zeros(len(train_df), dtype=float)
    p_grad_oof = np.zeros(len(train_df), dtype=float)

    for train_idx, valid_idx in skf.split(X_train, y_train):
        y_a = pd.Series(np.where(y_train.iloc[train_idx].to_numpy() == 'Dropout', 'Dropout', NOT_DROPOUT), index=y_train.index[train_idx])
        model_a = Pipeline(steps=[
            ('preprocessor', _dense_preprocessor(num_feats, cat_feats)),
            ('model', GradientBoostingClassifier(loss='log_loss', random_state=RANDOM_STATE)),
        ])
        model_a.set_params(**{f'model__{k}': v for k, v in params.items()})
        model_a.fit(X_train.iloc[train_idx], y_a, model__sample_weight=compute_sample_weight('balanced', y_a))
        p_not_dropout_oof[valid_idx] = _binary_proba(model_a, X_train.iloc[valid_idx], NOT_DROPOUT)

        stage_train_idx = [i for i in train_idx if y_train.iloc[i] != 'Dropout']
        y_b = y_train.iloc[stage_train_idx]
        model_b = Pipeline(steps=[
            ('preprocessor', _dense_preprocessor(num_feats, cat_feats)),
            ('model', GradientBoostingClassifier(loss='log_loss', random_state=RANDOM_STATE + 1000)),
        ])
        model_b.set_params(**{f'model__{k}': v for k, v in params.items()})
        model_b.fit(X_train.iloc[stage_train_idx], y_b, model__sample_weight=compute_sample_weight('balanced', y_b))
        p_grad_oof[valid_idx] = _binary_proba(model_b, X_train.iloc[valid_idx], 'Graduate')

    full_a = Pipeline(steps=[
        ('preprocessor', _dense_preprocessor(num_feats, cat_feats)),
        ('model', GradientBoostingClassifier(loss='log_loss', random_state=RANDOM_STATE)),
    ])
    full_a.set_params(**{f'model__{k}': v for k, v in params.items()})
    y_full_a = pd.Series(np.where(y_train.to_numpy() == 'Dropout', 'Dropout', NOT_DROPOUT), index=y_train.index)
    full_a.fit(X_train, y_full_a, model__sample_weight=compute_sample_weight('balanced', y_full_a))
    p_not_dropout_test = _binary_proba(full_a, X_test, NOT_DROPOUT)

    full_b = Pipeline(steps=[
        ('preprocessor', _dense_preprocessor(num_feats, cat_feats)),
        ('model', GradientBoostingClassifier(loss='log_loss', random_state=RANDOM_STATE + 1000)),
    ])
    full_b.set_params(**{f'model__{k}': v for k, v in params.items()})
    mask = y_train.to_numpy() != 'Dropout'
    y_full_b = y_train.loc[mask]
    full_b.fit(X_train.loc[mask], y_full_b, model__sample_weight=compute_sample_weight('balanced', y_full_b))
    p_grad_test = _binary_proba(full_b, X_test, 'Graduate')

    return _combine_binary(p_not_dropout_oof, p_grad_oof), _combine_binary(p_not_dropout_test, p_grad_test), y_test


def _weight_grid(names: Iterable[str], oof_probas: list[np.ndarray], y_true: np.ndarray):
    member_count = len(names)
    units = int(round(1.0 / WEIGHT_STEP))
    weights_names = list(names)
    best_score = -np.inf
    best_weight = None
    records: list[Dict[str, Any]] = []

    def rec(prefix, remain, left):
        if left == 1:
            yield prefix + [remain]
            return
        for v in range(remain + 1):
            yield from rec(prefix + [v], remain - v, left - 1)

    for weight_units in rec([], units, member_count):
        w = np.array(weight_units, dtype=float) / units
        pred = np.tensordot(w, np.stack(oof_probas, axis=0), axes=(0, 0))
        y_pred = np.array(LABELS)[np.argmax(pred, axis=1)]
        f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
        row = {f'weight_{name}': float(v) for name, v in zip(weights_names, w)}
        row['oof_f1_macro'] = float(f1)
        records.append(row)
        if f1 > best_score:
            best_score = f1
            best_weight = w

    return best_weight, float(best_score), pd.DataFrame(records).sort_values('oof_f1_macro', ascending=False)


def run_cumulative_link_reproduction(cv: int = 5, n_jobs: int = 16):
    train_df, test_df = load_processed_data()
    y_train = train_df[TARGET_COL].astype(str)
    y_test = test_df[TARGET_COL].astype(str)

    members = [
        ('catboost_cumulative_test', 'catboost', load_best_params('catboost_tuned')),
        ('lightgbm_hist_cumulative_test', 'lightgbm', load_best_params('hist_column_sampling')),
        ('gbdt_cumulative_test', 'gbdt', load_best_params('gbdt_log_loss')),
    ]

    member_names: list[str] = []
    oof_probas: list[np.ndarray] = []
    test_probas: list[np.ndarray] = []
    metric_rows: list[Dict[str, float]] = []
    stage_rows: list[Dict[str, float]] = []

    for name, kind, params in members:
        if kind == 'catboost':
            oof, test, y_stage = fit_predict_cum_catboost(train_df, test_df, params, cv)
        elif kind == 'lightgbm':
            oof, test, y_stage = fit_predict_cum_lightgbm(train_df, test_df, params, cv, n_jobs)
        else:
            oof, test, y_stage = fit_predict_cum_gbdt(train_df, test_df, params, cv)

        member_names.append(name)
        oof_probas.append(oof)
        test_probas.append(test)

        metric_rows.append(_eval_prob(f'{name}_oof', y_train, oof))
        metric_rows.append(_eval_prob(f'{name}_test', y_test, test))

        stage_rows.append(_eval_stage_a(f'{name}_stage_a_dropout_vs_not', y_stage, test))
        stage_rows.append(_eval_stage_b(f'{name}_stage_b_enrolled_vs_graduate', y_stage, test))

    best_w, best_f1, weight_grid = _weight_grid(member_names, oof_probas, y_train.to_numpy())
    oof_ensemble = np.tensordot(best_w, np.stack(oof_probas, axis=0), axes=(0, 0))
    test_ensemble = np.tensordot(best_w, np.stack(test_probas, axis=0), axes=(0, 0))

    metric_rows.append(_eval_prob('cumulative_link_ensemble_oof', y_train, oof_ensemble))
    metric_rows.append(_eval_prob('cumulative_link_ensemble_test', y_test, test_ensemble))

    metrics_df = pd.DataFrame(metric_rows)
    stage_df = pd.DataFrame(stage_rows)
    payload = {
        'members': member_names,
        'weights': {name: float(w) for name, w in zip(member_names, best_w)},
        'oof_f1_macro': best_f1,
        'weight_step': WEIGHT_STEP,
        'cv_splits': cv,
    }

    metrics_df.to_csv(OUT_DIR / 'cumulative_link_reproduction_metrics.csv', index=False)
    stage_df.to_csv(OUT_DIR / 'cumulative_link_reproduction_stage_metrics.csv', index=False)
    (OUT_DIR / 'cumulative_link_reproduction_best_weights.json').write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
    return metrics_df, stage_df, payload


In [ ]:
def fit_predict_catboost_multiclass(train_df: pd.DataFrame, test_df: pd.DataFrame, params: Dict[str, Any]):
    X_train, y_train, cat_idx = prepare_catboost_xy(train_df)
    X_test, y_test, _ = prepare_catboost_xy(test_df)
    model = CatBoostClassifier(
        loss_function='MultiClass',
        eval_metric='TotalF1:average=Macro',
        auto_class_weights='Balanced',
        thread_count=-1,
        allow_writing_files=False,
        random_seed=RANDOM_STATE,
        verbose=0,
        **params,
    )
    model.fit(X_train, y_train, cat_features=cat_idx)
    return _eval_prob('catboost', y_train, model.predict_proba(X_train)), _eval_prob('catboost_test', y_test, model.predict_proba(X_test))


def fit_predict_lightgbm_multiclass(train_df: pd.DataFrame, test_df: pd.DataFrame, params: Dict[str, Any]):
    X_train, y_train, cat_cols = prepare_lightgbm_xy(train_df)
    X_test, y_test, _ = prepare_lightgbm_xy(test_df)
    model = LGBMClassifier(
        objective='multiclass',
        class_weight='balanced',
        random_state=RANDOM_STATE,
        verbosity=-1,
        n_jobs=16,
        **params,
    )
    model.fit(X_train, y_train, categorical_feature=cat_cols)
    return _eval_prob('lightgbm', y_train, model.predict_proba(X_train)), _eval_prob('lightgbm_test', y_test, model.predict_proba(X_test))


def fit_predict_gbdt_multiclass(train_df: pd.DataFrame, test_df: pd.DataFrame, params: Dict[str, Any]):
    X_train = train_df.drop(columns=[TARGET_COL])
    y_train = train_df[TARGET_COL].astype(str)
    X_test = test_df.drop(columns=[TARGET_COL])
    y_test = test_df[TARGET_COL].astype(str)
    num_feats, cat_feats = split_feature_columns(X_train.columns)
    model = Pipeline(steps=[
        ('preprocessor', _dense_preprocessor(num_feats, cat_feats)),
        ('model', GradientBoostingClassifier(loss='log_loss', random_state=RANDOM_STATE)),
    ])
    model.set_params(**{f'model__{k}': v for k, v in params.items()})
    model.fit(X_train, y_train, model__sample_weight=compute_sample_weight('balanced', y_train))
    return _eval_prob('gbdt_log_loss', y_train, model.predict_proba(X_train)), _eval_prob('gbdt_log_loss_test', y_test, model.predict_proba(X_test))


def run_target_model_reproduction(target_model: str | None = None, cv: int = 5, n_jobs: int = 16):
    target = choose_target_row(target_model, TARGET_SET)
    target_name = str(target['Model'])

    expected_f1 = float(target['F1 (macro)'])
    expected_auc = target.get('AUC')

    train_df, test_df = load_processed_data()
    y_train = train_df[TARGET_COL].astype(str)
    y_test = test_df[TARGET_COL].astype(str)

    if any(token in target_name for token in ['cumulative', 'catboost_cumulative', 'lightgbm_hist_cumulative', 'gbdt_cumulative']):
        metrics_df, stage_df, payload = run_cumulative_link_reproduction(cv=cv, n_jobs=n_jobs)

        if target_name in set(metrics_df['model']):
            selected = metrics_df[metrics_df['model'] == target_name].iloc[0]
        elif target_name in set(stage_df['model']):
            selected = stage_df[stage_df['model'] == target_name].iloc[0]
        else:
            selected = metrics_df.sort_values('f1_macro', ascending=False).iloc[0]

        reproduced_f1 = float(selected.get('f1_macro', np.nan))
        reproduced_auc = float(selected.get('auc', np.nan))
        summary = {
            'target_model': target_name,
            'target_set': TARGET_SET,
            'expected_f1_macro': expected_f1,
            'reproduced_f1_macro': reproduced_f1,
            'delta_f1_macro': reproduced_f1 - expected_f1,
            'expected_auc': None if pd.isna(expected_auc) else float(expected_auc),
            'reproduced_auc': reproduced_auc,
            'matched_source': 'stage' if target_name in set(stage_df['model']) else 'member_or_ensemble',
            'reproduction_passed': bool(np.isfinite(reproduced_f1) and np.isfinite(expected_f1) and abs(reproduced_f1 - expected_f1) <= 1e-6),
            'best_ensemble_weights': payload['weights'],
        }
        (OUT_DIR / 'reproduction_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
        return summary, metrics_df, stage_df

    if target_name.startswith('catboost'):
        params = load_best_params('catboost_tuned')
        train_m, test_m = fit_predict_catboost_multiclass(train_df, test_df, params)
    elif target_name == 'lightgbm' or target_name.startswith('lightgbm_hist') or target_name.startswith('lightgbm_tuned'):
        params = load_best_params('hist_column_sampling' if (target_name == 'lightgbm' or target_name.startswith('lightgbm_hist')) else 'lightgbm_tuned')
        train_m, test_m = fit_predict_lightgbm_multiclass(train_df, test_df, params)
    elif target_name.startswith('gbdt_log_loss'):
        params = load_best_params('gbdt_log_loss')
        train_m, test_m = fit_predict_gbdt_multiclass(train_df, test_df, params)
    else:
        raise NotImplementedError(f"Model '{target_name}' is not supported in this notebook.")

    reproduced_f1 = float(test_m['f1_macro'])
    reproduced_auc = float(test_m['auc'])
    summary = {
        'target_model': target_name,
        'target_set': TARGET_SET,
        'expected_f1_macro': expected_f1,
        'reproduced_f1_macro': reproduced_f1,
        'delta_f1_macro': reproduced_f1 - expected_f1,
        'expected_auc': None if pd.isna(expected_auc) else float(expected_auc),
        'reproduced_auc': reproduced_auc,
        'reproduction_passed': bool(np.isfinite(reproduced_f1) and np.isfinite(expected_f1) and abs(reproduced_f1 - expected_f1) <= 1e-6),
    }
    (OUT_DIR / 'reproduction_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
    return summary, pd.DataFrame([train_m, test_m]), None


In [ ]:
# Run the default best-testing model automatically.
best_row = choose_target_row(None, TARGET_SET)
print('Project root:', PROJECT_ROOT)
print('Auto-selected target model:', best_row['Model'])
print('Auto-selected F1 (macro):', float(best_row['F1 (macro)']))

summary, metrics_df, stage_df = run_target_model_reproduction(target_model=best_row['Model'], cv=5, n_jobs=16)
print('\nReproduction summary:')
print(json.dumps(summary, ensure_ascii=False, indent=2))
print('\nTop reproducible Testing rows from this run:')
print(metrics_df.sort_values('f1_macro', ascending=False).head(20).to_string(index=False))
if stage_df is not None and not stage_df.empty:
    print('\nCumulative-link stage rows from this run:')
    print(stage_df.to_string(index=False))
print('\nOutput files:', OUT_DIR)


### Run any supported model separately
Set `TARGET_MODEL` to a model name and execute the cell below.


In [ ]:
TARGET_MODEL = None  # e.g., 'catboost_cumulative_test_stage_a_dropout_vs_not'

if TARGET_MODEL:
    summary, metrics_df, stage_df = run_target_model_reproduction(target_model=TARGET_MODEL, cv=5, n_jobs=16)
    print(f"Reproduced model: {TARGET_MODEL}")
    print(json.dumps(summary, ensure_ascii=False, indent=2))
else:
    print('TARGET_MODEL is None; default pipeline already run above.')


### Run all supported models from the embedded Testing list sequentially
Each model is executed in a run loop in this notebook.
Set `BATCH_RUN_ALL_MODELS=True` to start.


In [ ]:
def is_supported_model_name(model_name: str) -> bool:
    m = str(model_name).strip()
    if not m:
        return False
    if 'cumulative' in m:
        return True
    if m.startswith('catboost'):
        return True
    if m.startswith('lightgbm'):
        return True
    if m.startswith('gbdt_log_loss'):
        return True
    return False


def list_supported_testing_models(target_set: str = TARGET_SET, limit: int | None = None, rank_metric: str = 'F1 (macro)'):
    df = EMBEDDED_TESTING_MODELS_DF.copy()
    subset = df[df['Set'] == target_set].copy()
    if subset.empty:
        raise RuntimeError(f'No rows in Set={target_set}')
    subset[rank_metric] = pd.to_numeric(subset[rank_metric], errors='coerce')
    ordered = subset.sort_values(rank_metric, ascending=False)
    models: list[str] = []
    for m in ordered['Model'].astype(str):
        if m not in models and is_supported_model_name(m):
            models.append(m)
    if limit is not None and limit > 0:
        models = models[:limit]
    return models


def run_supported_testing_models(
    target_set: str = TARGET_SET,
    cv: int = 5,
    n_jobs: int = 16,
    limit: int | None = None,
    override_models: list[str] | None = None,
):
    if override_models is None:
        models = list_supported_testing_models(target_set=target_set, limit=limit)
    else:
        models = [str(m).strip() for m in override_models if str(m).strip()]
    print(f'Preparing to run {len(models)} supported Testing models.')
    records: list[dict[str, object]] = []

    for idx, model_name in enumerate(models, start=1):
        print(f'[{idx:03d}/{len(models)}] Running: {model_name}')
        try:
            summary, metrics_df, stage_df = run_target_model_reproduction(target_model=model_name, cv=cv, n_jobs=n_jobs)
            result_row = {
                'target_model': summary.get('target_model'),
                'status': 'ok',
                'expected_f1_macro': summary.get('expected_f1_macro'),
                'reproduced_f1_macro': summary.get('reproduced_f1_macro'),
                'delta_f1_macro': summary.get('delta_f1_macro'),
                'expected_auc': summary.get('expected_auc'),
                'reproduced_auc': summary.get('reproduced_auc'),
                'reproduction_passed': summary.get('reproduction_passed', False),
                'error': '',
                'stage_file': '',
            }
            if stage_df is not None and not stage_df.empty:
                stage_out = OUT_DIR / f"stage_metrics_{str(model_name).replace('/', '_').replace(' ', '_')}.csv"
                stage_df.to_csv(stage_out, index=False)
                result_row['stage_file'] = str(stage_out)
        except Exception as exc:
            print(f'FAILED: {model_name} -> {exc}')
            result_row = {
                'target_model': model_name,
                'status': 'failed',
                'expected_f1_macro': None,
                'reproduced_f1_macro': None,
                'delta_f1_macro': None,
                'expected_auc': None,
                'reproduced_auc': None,
                'reproduction_passed': False,
                'error': str(exc),
                'stage_file': '',
            }
        records.append(result_row)

    result_df = pd.DataFrame(records)
    result_path = OUT_DIR / 'all_supported_testing_models_batch_reproduction.csv'
    result_df.to_csv(result_path, index=False)
    print(f'Batch summary saved to: {result_path}')
    print(result_df[['target_model', 'status', 'expected_f1_macro', 'reproduced_f1_macro', 'delta_f1_macro', 'reproduction_passed', 'error']].to_string(index=False))
    return result_df


BATCH_RUN_ALL_MODELS = False
BATCH_RUN_LIMIT = None  # set integer if you only want top-K supported Testing models, e.g. 10

if BATCH_RUN_ALL_MODELS:
    batch_result_df = run_supported_testing_models(target_set=TARGET_SET, cv=5, n_jobs=16, limit=BATCH_RUN_LIMIT)
else:
    print('Set BATCH_RUN_ALL_MODELS=True to run all supported Testing models.')
